# ERK-KTR Full FOV Stimulation Pipeline
This notebook showcases how to use the ERK-KTR full FOV stimulation pipeline. The pipeline is designed to simulate the full field of view (FOV) stimulation of a cells with the ERK-KTR biosensor. As it is a demo experiment, the pipeline runs on the demo hardware provided by MicroManager. 

### Import required libraries

In [5]:
import os
import time
from rtm_pymmcore.data_structures import Fov, Channel, StimTreatment
from pprint import pprint
import pandas as pd
import numpy as np
import dataclasses
import random

### Experimental Settings

In [6]:
from rtm_pymmcore.microscope.MMDemo import MMDemo

mic = MMDemo()
mic.mmc.setChannelGroup("Channel")

# # for Jungfrau Microscope enable here:
# from rtm_pymmcore.microscope.Jungfrau import Jungfrau

# mic = Jungfrau()
# mic.mmc.setChannelGroup("TTL_ERK")

In [7]:
## Configuration options
FIRST_FRAME_STIMULATION = 10
# N_FRAMES = 340
N_FRAMES = 4


SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

TIME_BETWEEN_TIMESTEPS = 5  # time in seconds between frames
TIME_PER_FOV = 3.3  # time in seconds per fov

MIX_STIM_EXPOSURE = False  # if True, the stim_exposure list will be randomized
ADD_STIM_EXPOSURE_GROUP = (
    False  # add an entry showing the last stimulation exposure for each FOV
)
REGULAR_SPACING_BETWEEN_STIMULATIONS = (
    False  # if True, the stim_timesteps will be evenly spaced
)


## Storage path for the experiment
base_path = "C:\\Users\\Alex\\Ausbildung\\PhD_temp\\test_exp"
experiment_name = "exp_test"
path = os.path.join(base_path, experiment_name)

path_with_old_data_for_simulation = os.path.join(
    ".", "test_exp_data", "00_01_demo_imgs"
)  # path to the folder with old data for simulation


# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
channels = []
channels.append(Channel(name="miRFP", exposure=300))
channels.append(Channel(name="mScarlet3", exposure=200))

# Channel to check for the expression of the optogenetic marker, can be used if it the marker is in the same channel as the stimulation channel.
channel_optocheck = Channel(name="mCitrine", exposure=300)
optocheck_timepoints = (0, 1)  # timepoints in frames when optocheck is done


condition = ["test"]

# condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24 # Example of adding multiple conditions to the dataframe. n repreats the amount of times the condition is repeated.


n_fovs_per_well = None  ## change this variable to the amount of fovs that you have per well. Set to None if you are not working with wellplate.


# Stimulation parameters for optogenetics. The stimulation will be repeated for each condition.

stim_exposures_timesteps = [
    {
        "ramp_pattern_name": "Sustained",
        "stim_exposure_list": (250,) * 300,
        "stim_timestep": range(10, 311, 1),
    }
]

stim_exposures_timesteps = [
    {
        "ramp_pattern_name": "Sustained",
        "stim_exposure_list": (250,) * 3,
        "stim_timestep": (0, 1, 2),
    }
]


for stim_exposure_timestep in stim_exposures_timesteps:
    if isinstance(stim_exposure_timestep["stim_timestep"], range):
        stim_exposure_timestep["stim_timestep"] = tuple(
            stim_exposure_timestep["stim_timestep"]
        )
    if isinstance(stim_exposure_timestep["stim_exposure_list"], range):
        stim_exposure_timestep["stim_exposure_list"] = tuple(
            stim_exposure_timestep["stim_exposure_list"]
        )
    if isinstance(stim_exposure_timestep["stim_timestep"], list):
        stim_exposure_timestep["stim_timestep"] = tuple(
            stim_exposure_timestep["stim_timestep"]
        )
    if isinstance(stim_exposure_timestep["stim_exposure_list"], list):
        stim_exposure_timestep["stim_exposure_list"] = tuple(
            stim_exposure_timestep["stim_exposure_list"]
        )


if MIX_STIM_EXPOSURE:
    for stim_exposure_timestep in stim_exposures_timesteps:
        stim_exp_list = list(stim_exposure_timestep["stim_exposure_list"])
        random.shuffle(stim_exp_list)
        stim_exposure_timestep["stim_exposure_list"] = tuple(stim_exp_list)

for stexptim in stim_exposures_timesteps:
    print("Pattern Name: ", stexptim["ramp_pattern_name"])

    for stim_exp, stim_timestep in zip(
        stexptim["stim_exposure_list"], stexptim["stim_timestep"]
    ):
        print(f"{stim_exp} at {stim_timestep}")
    print("")

    # add general stimulation parameters
    stexptim["stim_power"] = 10
    stexptim["stim_channel_name"] = "CyanStim"
    stexptim["stim_channel_group"] = "TTL_ERK"
    stexptim["stim_channel_device_name"] = "Spectra"
    stexptim["stim_channel_power_property_name"] = "Cyan_Level"


## Define the Tools that you are using for the experiment
from rtm_pymmcore.segmentation.base_segmentation import DummySegmentator
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.simple_fe import SimpleFE

segmentators = None
stimulator = None
feature_extractor = None
tracker = None
optocheck = None


from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck,
)
mic.set_pipeline(pipeline=pipeline)

Pattern Name:  Sustained
250 at 0
250 at 1
250 at 2

Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\raw already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\tracks already exists


### GUI - Napari Micromanager

#### Load GUI

In [4]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

c:\Users\Jungfrau\Documents\alandolt\code\rtm-pymmcore\.venv\Lib\site-packages\napari_micromanager\main_window.py:51: FutureWarning: The `_dock_widgets` property is private and should not be used in any plugin code. Please use the `dock_widgets` property instead.
  if "MinMax" not in getattr(self.viewer.window, "_dock_widgets", []):


Please execute the following code block, if you would like to set a custom ROI (e.g. smaller illumination than the full field of view of the camera). Execute it after you have started the camera once in the GUI. 

In [ ]:
if mic.SET_ROI_REQUIRED:
    mic.mmc.clearROI()
    mic.mmc.setROI(mic.ROI_X, mic.ROI_Y, mic.ROI_WIDTH, mic.ROI_HEIGHT)

### Map Experiment to FOVs

#### If FOVs already saved - Reload them from file

In [8]:
import json

# file = os.path.join(path, "fovs.json")
file = os.path.join("fovs.json")
with open(file, "r") as f:
    data_mda_fovs = json.load(f)
data_mda_fovs = data_mda_fovs
load_from_file = True

### Use FOVs to generate dataframe for acquisition

In [9]:
n_fovs_simultaneously = TIME_BETWEEN_TIMESTEPS // TIME_PER_FOV
timesteps = range(N_FRAMES)

start_time = 0
if not load_from_file:
    data_mda_fovs = viewer.window._dock_widgets["MDA"].widget().value().stage_positions
    data_mda_fovs_dict = []
    for data_mda in data_mda_fovs:
        data_mda_fovs_dict.append(data_mda.model_dump())
    data_mda_fovs = data_mda_fovs_dict
    if data_mda_fovs is None:
        assert False, "No fovs selected. Please select fovs in the MDA widget"

dfs = []
fovs = []
sub_experiment = 0

for fov_index, fov in enumerate(data_mda_fovs):
    fov_object = Fov(fov_index)
    fovs.append(fov_object)
    fov_group = fov_index // n_fovs_simultaneously
    start_time = fov_group * TIME_BETWEEN_TIMESTEPS * len(timesteps)
    if len(condition) == 1:
        condition_fov = condition[0]
    else:
        condition = condition[fov_index // n_fovs_per_condition]
    for timestep in timesteps:
        row = {
            "fov_object": fov_object,
            "fov": fov_index,
            "fov_x": fov.get("x"),
            "fov_y": fov.get("y"),
            "fov_z": fov.get("z") if not mic.USE_ONLY_PFS else None,
            "fov_name": str(fov_index) if fov["name"] is None else fov["name"],
            "timestep": timestep,
            "time": start_time + timestep * TIME_BETWEEN_TIMESTEPS,
            "cell_line": condition_fov,
            "channels": tuple(dataclasses.asdict(channel) for channel in channels),
            "fname": f"{str(fov_index).zfill(3)}_{str(sub_experiment).zfill(2)}_{str(timestep).zfill(5)}",
        }
        dfs.append(row)

df_acquire = pd.DataFrame(dfs)

print(f"Total Experiment Time: {df_acquire['time'].max()/3600}h")


df_acquire = df_acquire.dropna(axis=1, how="all")
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", True)
df_acquire = df_acquire.sort_values(by=["time", "fov"])
df_acquire

Total Experiment Time: 0.009722222222222222h


,fov_object,fov,fov_x,fov_y,fov_z,fov_name,timestep,time,cell_line,channels,fname
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,0.00,0.0,0.0,0,0,0.0,test,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00000
1,<rtm_pymmcore.data_structures.Fov object at 0x...,0,0.00,0.0,0.0,0,1,5.0,test,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00001
2,<rtm_pymmcore.data_structures.Fov object at 0x...,0,0.00,0.0,0.0,0,2,10.0,test,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00002
3,<rtm_pymmcore.data_structures.Fov object at 0x...,0,0.00,0.0,0.0,0,3,15.0,test,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00003
4,<rtm_pymmcore.data_structures.Fov object at 0x...,1,20.01,0.0,0.0,1,0,20.0,test,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00000
5,<rtm_pymmcore.data_structures.Fov object at 0x...,1,20.01,0.0,0.0,1,1,25.0,test,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00001
6,<rtm_pymmcore.data_structures.Fov object at 0x...,1,20.01,0.0,0.0,1,2,30.0,test,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00002
7,<rtm_pymmcore.data_structures.Fov object at 0x...,1,20.01,0.0,0.0,1,3,35.0,test,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00003


In [10]:
df_acquire_post_pause = []
previous_df = df_acquire
for fov_index in previous_df["fov"].unique():
    fov_object = previous_df[previous_df["fov"] == fov_index]["fov_object"].iloc[0]
    old_data_fov = previous_df[previous_df["fov"] == fov_index].iloc[0]
    row = {
        "fov_object": fov_object,
        "fov": fov_index,
        "fov_x": old_data_fov["fov_x"],
        "fov_y": old_data_fov["fov_y"],
        "fov_z": old_data_fov["fov_z"],
        "fov_name": old_data_fov["fov_name"],
        "timestep": None,
        "time": None,
        "cell_line": old_data_fov["cell_line"],
        "channels": None,
        "fname": None,
    }
    df_acquire_post_pause.append(row)
df_acquire_post_pause = pd.DataFrame(df_acquire_post_pause)
df_acquire_post_pause

,fov_object,fov,fov_x,fov_y,fov_z,fov_name,timestep,time,cell_line,channels,fname
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,0.00,0.0,0.0,0,None,None,test,None,None
1,<rtm_pymmcore.data_structures.Fov object at 0x...,1,20.01,0.0,0.0,1,None,None,test,None,None


### Run experiment

In [11]:
for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()

except:
    pass


mic.run_experiment(df_acquire)

[04.11.2025 09:48:40] WARNING  Failed to set channel. Preset "miRFP" of configuration group          ]8;id=502569;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py\_engine.py]8;;\:]8;id=671677;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py#822\822]8;;\
                               "Channel" does not exist                                                            

                      WARNING  Failed to set channel. Preset "mScarlet3" of configuration group      ]8;id=770168;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py\_engine.py]8;;\:]8;id=856513;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py#822\822]8;;\
                               "Channel" does not exist                                                            

[04.11.2025 09:48:45] WARNING  Failed to set channel. Preset "miRFP" of configuration group          ]8;id=525473;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py\_engine.py]8;;\:]8;id=457816;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py#822\822]8;;\
                               "Channel" does not exist                                                            

                      WARNING  Failed to set channel. Preset "mScarlet3" of configuration group      ]8;id=479956;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py\_engine.py]8;;\:]8;id=613426;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py#822\822]8;;\
                               "Channel" does not exist                                                            

[04.11.2025 09:48:50] WARNING  Failed to set channel. Preset "miRFP" of configuration group          ]8;id=177271;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py\_engine.py]8;;\:]8;id=595033;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py#822\822]8;;\
                               "Channel" does not exist                                                            

...

                      WARNING  Failed to set channel. Preset "mScarlet3" of configuration group      ]8;id=114550;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py\_engine.py]8;;\:]8;id=592274;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py#822\822]8;;\
                               "Channel" does not exist                                                            

[04.11.2025 09:49:10] WARNING  Failed to set channel. Preset "miRFP" of configuration group          ]8;id=706275;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py\_engine.py]8;;\:]8;id=980756;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py#822\822]8;;\
                               "Channel" does not exist                                                            

                      WARNING  Failed to set channel. Preset "mScarlet3" of configuration group      ]8;id=396417;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py\_engine.py]8;;\:]8;id=103616;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py#822\822]8;;\
                               "Channel" does not exist                                                            

[04.11.2025 09:49:15] WARNING  Failed to set channel. Preset "miRFP" of configuration group          ]8;id=892270;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py\_engine.py]8;;\:]8;id=732838;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py#822\822]8;;\
                               "Channel" does not exist                                                            

                      WARNING  Failed to set channel. Preset "mScarlet3" of configuration group      ]8;id=525747;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py\_engine.py]8;;\:]8;id=219930;file://c:\Users\Alex\Programmierung\01_git\PhD\rtm-pymmcore\.venv\Lib\site-packages\pymmcore_plus\mda\_engine.py#822\822]8;;\
                               "Channel" does not exist                                                            

In [12]:
mic.run_experiment(df_acquire_post_pause)

In [ ]:
mic.post_experiment()
time.sleep(10)


fovs_i_list = os.listdir(os.path.join(path, "tracks"))
fovs_i_list.sort()
dfs = []

for fov_i in fovs_i_list:

    track_file = os.path.join(path, "tracks", fov_i)
    df = pd.read_parquet(track_file)
    dfs.append(df)

pd.concat(dfs).to_parquet(os.path.join(path, "exp_data.parquet"))

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

### Function to re-connect link with GUI if manually broken

The following functions can be used to manually re-make the connection between the GUI and the running rtm-pymmcore script. However, normally you don't need to execute them. 

In [ ]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

The following code block can be used to manually break the connection between GUI and Jupyter Notebook:


In [ ]:
### Break connection
# mm_wdg._core_link.cleanup()